# **Projection correction for Nearest Neighbour (NN) or Minimum Spanning Tree (MST) distances**

These correction factors follow the empirical fit described in *Barnes et al. 2026*.

Returns fitted parameters in form: $$C(N, SDR) = C_\infty * [1 - \exp(-SDR / S_0)] * (N / N_0)^\beta.$$

Fitting this form to the grid of ensemble-averaged measurements from 1000 fractal realisations with $D=1.7$-$2.5$, $n_{\mathrm{div}}=2$-$4$ yields (see Fig 8 and Sect 4 of Barnes et al. 2026 for full discussion):
\begin{align*}
\mathcal{C}_\infty &= 1.94 \pm 0.01,\\
S_0      &= 21.8 \pm 0.3,\\
\beta    &= 0.173 \pm 0.003.
\end{align*}

Use `projection_correction` to compute the fitted correction factor C(N, SDR) and apply it to your own measurements.

- Accepts scalars, lists, or numpy arrays.
- Parameters can come from the packaged YAML file, a custom YAML file, or explicit values.


In [1]:
# Core scientific imports
import numpy as np
from pathlib import Path

# corespaceing3d imports
from corespaceing3d import projection_correction

## Basic usage

This uses the default packaged fit parameters (if available).

In [2]:
C = projection_correction(N=100, SDR=50)
print(f"C(N=100, SDR=50) = {C:.3f}")

C(N=100, SDR=50) = 1.743


## Vectorized usage (broadcasting)


In [3]:
N = np.array([5, 10, 20, 50, 100, 200], dtype=float)
SDR = np.array([5, 10, 20, 50, 100, 200, 500], dtype=float)

C_grid = projection_correction(N[:, None], SDR[None, :])
print("C grid shape:", C_grid.shape)
print(np.round(C_grid, 2))

C grid shape: (6, 7)
[[0.24 0.42 0.69 1.04 1.14 1.16 1.16]
 [0.27 0.48 0.78 1.17 1.29 1.3  1.3 ]
 [0.3  0.54 0.88 1.32 1.45 1.47 1.47]
 [0.35 0.63 1.03 1.55 1.7  1.72 1.72]
 [0.4  0.71 1.16 1.74 1.92 1.94 1.94]
 [0.45 0.8  1.31 1.96 2.16 2.19 2.19]]


## Apply correction to your projected data

Here we correct example projected mean nearest-neighbor lengths.


In [6]:
mean_nn_2d = np.array([0.21, 0.18, 0.15])
N_vals = np.array([50, 100, 200], dtype=float)
SDR_vals = np.array([50, 100, 200], dtype=float)

mean_nn_3d_est, C_vals = projection_correction(
    N_vals,
    SDR_vals,
    apply_to=mean_nn_2d,
    return_factor=True,
)

print("C:", np.round(C_vals, 3))
print("Corrected mean_nn_3d:", np.round(mean_nn_3d_est, 3))

C: [1.546 1.919 2.185]
Corrected mean_nn_3d: [0.325 0.345 0.328]


## Load parameters from a specific file or pass explicit values

Swap `params_path` to any fitted YAML you have (for example, `fit_params_C_vs_N_SDR.yaml`).


In [5]:
params_path = Path("../data/products/fit_params_C_vs_N_SDR_Barnes2025.yaml")
C_from_file = projection_correction(N=100, SDR=50, params_path=params_path)
print(f"C from file = {C_from_file:.3f}")

C_from_params = projection_correction(
    N=100,
    SDR=50,
    C_inf=1.94,
    S0=22.0,
    beta=0.173,
    N0=100.0,
)
print(f"C from explicit params = {C_from_params:.3f}")

C from file = 1.743
C from explicit params = 1.740
